# 00 · Problem definition

This notebook fixes **what is being solved, how it will be measured, and what
counts as success** — all of it before a single model is trained.

### A note on execution order

Although it is numbered first, it was written **after** notebook 02. That is not
an oversight: exploratory analysis *has* to inform the framing. Fixing business
metrics without having looked at the data produces invented criteria.

What is non-negotiable is that **no model has been trained yet**. The acceptance
criteria in section 8 and the cut-off method in section 9 are registered here and
are not touched afterwards. That is the distinction that matters: the EDA informs
the framing, the model results do not.

> Dependency: the business baseline is computed from `silver.customers_clean`, so
> on the first pass this notebook runs after 02.

In [6]:
import sys
sys.path.append("../src")

from config import bootstrap

ctx = bootstrap()
spark, w = ctx.spark, ctx.w

connected to Databricks
  branch   : sandbox
  catalog  : bank_churn_eng
  identity : juzoushio@...


## 1 · Context

A bank operating in **France, Germany and Spain** is losing customers steadily. The
base analysed holds **10,000 customers**, of whom **2,037 (20.4%) have left**.

The problem is not the rate itself. It is that the bank **does not know who it is
about to lose**. Without that, there are two options and both are bad: do nothing,
or run indiscriminate campaigns that spend budget on customers who were never
going to leave.

Retention is substantially cheaper than acquisition. A lost customer takes their
recurring margin with them and forces spending to replace it.

What changes here is the move from a retrospective diagnosis — "we lost 2,037
customers" — to an anticipatory one: "these are the customers we will probably
lose, and these are the actions that make sense". The goal is not prediction for
its own sake. It is **prioritising a limited retention budget**.

## 2 · Who consumes this

| Role | What they need | How they consume it |
|---|---|---|
| **Retention management** | Who to contact this month on a fixed budget | Ranked list with a suggested action |
| **Campaigns / CRM** | Actionable segments, not abstract scores | `customer_predictions` in the operational system |
| **Country commercial leads** | Why Germany churns at twice the rate | Segment analysis and EDA findings |
| **Risk and compliance** | A model that neither discriminates nor is unauditable | Bias analysis and traceability per prediction |
| **Data team** | A reproducible, maintainable pipeline | Versioned notebooks and governed tables |

The **primary consumer is the campaigns team**, and that shapes the design: they
need a list of customers with an action attached, not a metrics report. It is why
the predictions travel back to the operational system instead of staying in the
lakehouse.

## 3 · Target and predictors

**Target:** `exited`, boolean. `True` means the customer left.

This is supervised binary classification. Worth naming what the dataset does *not*
contain: there is no churn date and no time horizon. We do not know whether
someone left last month or three years ago, so the model estimates a
**propensity**, not a risk over a defined period. That limitation carries into the
interpretation and is declared in section 10.

| Group | Variables | Note |
|---|---|---|
| **Target** | `exited` | What is predicted |
| **Demographic** | `age`, `gender`, `geography` | Sensitive: see the bias decision |
| **Product** | `num_of_products`, `has_cr_card`, `balance`, `tenure` | |
| **Behavioural** | `is_active_member` | **The only actionable one** |
| **Financial** | `credit_score`, `estimated_salary` | |
| **Derived** | `balance_zero`, `age_group`, `products_group` | Built in silver |
| **Traceability, never a predictor** | `customer_id` | To return the prediction to the CRM |
| **Excluded** | `row_number`, `surname` | Technical identifier and personal data |

The split between **actionable** and **non-actionable** predictors is not
decorative. Age, country and gender predict well and cannot be changed.
`is_active_member` is the only variable the bank can actually intervene on, which
makes it worth more per unit of predictive power than any of the others.

## 4 · Questions

> **Which customers are most likely to leave, what explains that risk, and which
> retention actions should be prioritised?**

| | Question | Why it matters |
|---|---|---|
| **Q1** | What is the overall churn rate and how imbalanced is it? | Drives metric choice and class handling |
| **Q2** | Which segments concentrate the risk, and how much of total churn do they explain? | A segment with 90% churn and 30 customers is irrelevant next to one with 35% and 2,000 |
| **Q3** | How does risk vary with age, activity, products, balance, geography and credit score? | With explicit attention to **non-monotonic** relationships |
| **Q4** | Which associations are statistically sound, and where can causality not be claimed? | Includes discarding apparent findings that turn out to be confounders |
| **Q5** | Does the model beat the rule of thumb the bank could apply with no model at all? | The question that decides whether the project adds value |
| **Q6** | Does performance change across gender or country groups? | A fairness requirement, and a condition for deployment |
| **Q7** | How many high-risk customers does the chosen threshold identify, and what does being wrong cost? | Translates the confusion matrix into euros |

## 5 · Technical metrics

### Why accuracy does not work here

With 20.4% churn, a model that predicts "nobody leaves" is right **79.6%** of the
time and is completely useless: it detects not one customer at risk. The next cell
demonstrates that numerically rather than asserting it.

| Metric | Why |
|---|---|
| **Recall, class 1** | *Primary.* A false negative is a customer lost without anyone trying |
| **Precision, class 1** | Every false positive consumes campaign budget |
| **F1, class 1** | Summary of the balance between the two above |
| **ROC-AUC** | Ranking ability, independent of the threshold |
| **Average precision** | More informative than ROC with imbalanced classes |
| **Confusion matrix** | The four numbers that translate into euros |

**Accuracy is reported only to document why it is discarded.**

The metric optimised during hyperparameter search is **average precision**, not
accuracy and not ROC-AUC. It is the one that reflects the real problem: ranking
the minority well.

In [7]:
from config import UC_CATALOG

SILVER_TABLE = f"{UC_CATALOG}.silver.customers_clean"

df = spark.table(SILVER_TABLE).toPandas()
df["exited"] = df["exited"].astype(bool)

n       = len(df)
n_churn = int(df["exited"].sum())
rate    = df["exited"].mean()

print(f"customers      : {n:,}")
print(f"churned        : {n_churn:,}")
print(f"churn rate     : {rate:.2%}")
print(f"imbalance      : 1 to {(1 - rate) / rate:.1f}")

print("\n--- the accuracy problem ---")
print(f"model 'nobody leaves' -> accuracy {1 - rate:.2%}, recall class 1 {0:.0%}")
print(f"                         customers at risk detected: 0 of {n_churn:,}")

customers      : 10,000
churned        : 2,037
churn rate     : 20.37%
imbalance      : 1 to 3.9

--- the accuracy problem ---
model 'nobody leaves' -> accuracy 79.63%, recall class 1 0%
                         customers at risk detected: 0 of 2,037


## 6 · Business metric and cost model

Technical metrics cannot be taken to a steering committee. They have to be
translated into euros, and that needs assumptions the dataset does not contain.

**They are declared explicitly as assumptions, not as data.** They live in
`src/config.py` as parameters: if the bank provides real figures, they are
substituted and the whole analysis recomputes.

| Parameter | Assumed | Reasoning |
|---|---|---|
| Annual margin per customer | 200 € | Typical order of magnitude in retail banking |
| Retention horizon | 3 years | Undiscounted, to avoid adding another assumption |
| Contact cost | 20 € | Commercial handling and channel |
| Incentive cost | 50 € | Only paid if the customer accepts |
| Campaign success rate | 30% | Of the leavers who get contacted |
| Operating capacity | 800 per campaign | The team's fixed budget |

From those six, two numbers fall out, and everything downstream is built on them.

In [8]:
from config import CLV, VALUE_TP, COST_FP, BREAK_EVEN, DECISION_UNIT, CAPACITY

print(f"assumed CLV                        : {CLV:,.0f} EUR")
print(f"expected value if they WERE leaving: {VALUE_TP:+,.0f} EUR   (true positive)")
print(f"expected value if they were NOT    : {-COST_FP:+,.0f} EUR   (false positive)")
print(f"benefit / cost ratio               : {VALUE_TP / COST_FP:.1f} to 1")

print(f"\nbreak-even probability: p* = {BREAK_EVEN:.4f}")
print(f"  below that, calling destroys value")
print(f"  far below 0.5 -- using 0.5 would exclude customers worth calling")

print(f"\nthe number to remember: {VALUE_TP:.0f} + {COST_FP:.0f} = {DECISION_UNIT:.0f} EUR")
print("  it is the denominator of the threshold and also what separates")
print("  a hit from a miss on a single call. The natural unit of this work.")

assumed CLV                        : 600 EUR
expected value if they WERE leaving: +145 EUR   (true positive)
expected value if they were NOT    : -35 EUR   (false positive)
benefit / cost ratio               : 4.1 to 1

break-even probability: p* = 0.1944
  below that, calling destroys value
  far below 0.5 -- using 0.5 would exclude customers worth calling

the number to remember: 145 + 35 = 180 EUR
  it is the denominator of the threshold and also what separates
  a hit from a miss on a single call. The natural unit of this work.


## 7 · The business baseline

Here is the criterion that decides whether the project is worth anything.

A `DummyClassifier` proves accuracy is useless, but it is a straw man. **The
model's real competition is what the bank can do for free in an afternoon**: a
rule based on the variable the EDA identified as dominant.

If a random forest tuned over two days does not clearly beat
`WHERE age BETWEEN 40 AND 59`, the project does not justify its cost. Almost no
churn project submits itself to this test — they all compare against random, which
is a bar anything clears.

In [9]:
base = (df.groupby("age_group")
          .agg(customers=("exited", "size"), churned=("exited", "sum"))
          .assign(rate=lambda d: (d.churned / d.customers * 100).round(1))
          .sort_values("rate", ascending=False))

base["pct_base"]  = (base.customers / n       * 100).round(1)
base["pct_churn"] = (base.churned   / n_churn * 100).round(1)
base["lift"]      = (base.pct_churn / base.pct_base).round(2)

print("risk by age band (from silver):")
print(base.to_string())

rule     = df["age_group"].isin(["40-49", "50-59"])
coverage = df.loc[rule, "exited"].sum() / n_churn
volume   = rule.mean()

print(f"\n--- BASELINE RULE: contact customers aged 40 to 59 ---")
print(f"customers contacted : {rule.sum():,} ({volume:.1%} of the base)")
print(f"churners reached    : {int(df.loc[rule, 'exited'].sum()):,} ({coverage:.1%} of total)")
print(f"lift                : {coverage / volume:.2f}x")
print(f"precision           : {df.loc[rule, 'exited'].mean():.1%}")

risk by age band (from silver):
           customers  churned  rate  pct_base  pct_churn  lift
age_group                                                     
50-59            869      487  56.0       8.7       23.9  2.75
40-49           2618      806  30.8      26.2       39.6  1.51
60+              526      147  27.9       5.3        7.2  1.36
30-39           4346      473  10.9      43.5       23.2  0.53
18-29           1641      124   7.6      16.4        6.1  0.37

--- BASELINE RULE: contact customers aged 40 to 59 ---
customers contacted : 3,487 (34.9% of the base)
churners reached    : 1,293 (63.5% of total)
lift                : 1.82x
precision           : 37.1%


In [10]:
# The same rule, capped at the real campaign capacity. Highest-risk bands are
# taken first until the quota runs out.
acc_customers = acc_churn = 0.0
selected = []

for band in base.index.tolist():
    row = base.loc[band]
    if acc_customers + row.customers <= CAPACITY:
        selected.append(band)
        acc_customers += row.customers
        acc_churn     += row.churned
    else:
        remaining = CAPACITY - acc_customers
        if remaining > 0:
            selected.append(f"{band} (partial: {int(remaining)})")
            acc_customers += remaining
            acc_churn     += remaining * row.rate / 100
        break

print(f"with {CAPACITY} contacts, the age rule selects: {selected}")
print(f"  contacted        : {int(acc_customers):,}")
print(f"  churners captured: {int(acc_churn):,} of {n_churn:,} ({acc_churn/n_churn:.1%})")

expected = acc_churn * VALUE_TP - (acc_customers - acc_churn) * COST_FP
print(f"  expected value   : {expected:+,.0f} EUR")
print("\nThis is the number the model has to beat.")

with 800 contacts, the age rule selects: ['50-59 (partial: 800)']
  contacted        : 800
  churners captured: 448 of 2,037 (22.0%)
  expected value   : +52,640 EUR

This is the number the model has to beat.


## 8 · Acceptance criteria

**Registered before any model is trained. They are not modified in light of the
results.**

### The main criterion

> With the **same contact volume**, the model must capture **more churners** than
> the age rule computed in section 7.

It is the only criterion that decides whether the project justifies its existence.
The rest are quality conditions.

### Technical

| Criterion | Threshold |
|---|---|
| ROC-AUC on test | ≥ 0.75 |
| Recall, class 1, at the chosen threshold | ≥ 0.60 |
| Precision, class 1 | ≥ 0.40 |
| Beat the `DummyClassifier` on recall | mandatory |

### Fairness

| Criterion | Threshold |
|---|---|
| Recall gap between gender groups | ≤ 10 points, or explicitly justified |
| Recall gap between countries | ≤ 10 points, or explicitly justified |
| Cost of excluding sensitive variables | **measured and reported**, not assumed |

### Reproducibility

- Fixed seed in every random process (`RANDOM_STATE = 42`)
- Stratified split
- Every transformer that learns statistics lives inside the `Pipeline`
- Experiments logged to MLflow

### What would count as failure

Worth saying out loud, because a criterion nobody can fail is not a criterion.
The project fails if the model does not beat the age rule at equal volume, if it
cannot be audited, or if a fairness gap appears that cannot be explained.

> **Spoiler, recorded honestly:** two of these eight were not met. The recall
> criterion turned out to be arithmetically impossible under a fixed capacity, and
> the country recall gap exceeded ten points. Neither was adjusted afterwards.
> Notebook 05 reports both as failures and explains why.

## 9 · Risk cut-off method — pre-registered

`risk_level` takes three values: `high`, `medium`, `low`. **The method for setting
the boundaries is decided here; the actual numbers are computed in notebook 05.**

| Level | Definition | Action |
|---|---|---|
| **high** | The `CAPACITY` customers with the highest predicted probability | Proactive contact with incentive |
| **medium** | Above the break-even threshold, outside the quota | Watch; include if budget remains |
| **low** | Everyone else | No action |

### Why operating capacity and not a fixed threshold

Because that is how the decision is actually made. The campaign budget is fixed,
and the business question is not "who is at risk?" but **"who do I call with the
resources I have?"**. A cut at 0.5 or 0.3 is arbitrary and can produce a list of
3,000 people when only 800 can be called.

The break-even threshold from section 6 acts as a **validation**: if the quota of
800 ends up including customers below it, contacting them would destroy value and
the quota should be trimmed.

### Why pre-register it

If the cuts are chosen **after** seeing the score distribution, the temptation —
conscious or not — is to pick the ones that make the model look good. That is
post-hoc rationalisation, and it is one of the first things an experienced
reviewer looks for.

## 10 · Declared assumptions and limitations

Declared now, not at the end of the report where they no longer inconvenience
anyone.

### About the data

**The dataset is synthetic.** Evidence accumulated during exploration: `age` and
`tenure` independent (ρ = −0.01), `estimated_salary` uniformly distributed
(kurtosis ≈ −1.2), and `num_of_products = 4` with 100% churn across 60 cases. This
implies a **structural ceiling**: the interactions a model can learn are limited to
the ones the generator introduced.

**There is no time dimension.** With no churn date and no signup date, the model
estimates propensity, not risk over a period. It cannot say "this customer will
leave within three months".

**The Germany effect is not explainable with this data.** It doubles the churn rate
while being indistinguishable on every observable variable. `geography` will act as
a proxy for an unknown factor.

**There is no product catalogue.** `num_of_products` is a count with no detail, so
the strongest pattern in the dataset ends up described but not explained.

### About the economics

The six figures in section 6 are **reasoned estimates, not the bank's data**. They
change the optimal threshold and the expected value, though not the ordering of
customers by risk. They should be replaced with real figures before any deployment.

### About causality

Everything reported will be an **association**. That inactive customers churn more
does not establish that inactivity causes churn: it could run the other way, or a
third factor could drive both.

---

## Summary

| | |
|---|---|
| **Problem** | Binary classification: predict `exited` |
| **Real objective** | Prioritise a limited retention budget |
| **Primary metric** | Recall, class 1 |
| **Optimisation metric** | Average precision |
| **Business metric** | Net expected campaign value |
| **Rival to beat** | The age rule, not random |
| **Risk cut-off** | Operating capacity, validated against the cost threshold |
| **Actionable variable** | `is_active_member` |

**Next:** `03_eda_analysis` — formalise with visualisation and inference the
findings that notebook 02 established numerically.